In [ ]:
# 在 Notebook Settings 中启用 GPU 和 Internet 后运行
!nvidia-smi

In [ ]:
# Kaggle Dataset 是只读输入；所有运行产物写入 /kaggle/working
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/MuVisual")
OUTPUT_DIR = WORK_ROOT / "output"
CACHE_DIR = WORK_ROOT / "cache"

for path in (OUTPUT_DIR, CACHE_DIR / "huggingface", CACHE_DIR / "torch", CACHE_DIR / "separator"):
    path.mkdir(parents=True, exist_ok=True)

audio_extensions = {".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a", ".aiff", ".ac3"}
audio_files = sorted(path for path in INPUT_DIR.rglob("*") if path.is_file() and path.suffix.lower() in audio_extensions)
assert audio_files, "/kaggle/input 中没有支持的音频；请先通过 Add Input 添加音频 Dataset"
print(f"发现 {len(audio_files)} 个音频文件")
for path in audio_files[:20]:
    print(path)

In [ ]:
# 下载或更新项目
import subprocess

repository = Path("/kaggle/working/MuVisual-Workflow")
if repository.exists():
    subprocess.run(["git", "-C", str(repository), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/TecReaGroup/MuVisual-Workflow.git", str(repository)], check=True)

%cd /kaggle/working/MuVisual-Workflow
!git rev-parse --short HEAD

In [ ]:
# 安装音频系统依赖
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1 build-essential
!ffmpeg -version | head -n 1

In [ ]:
# 安装 uv、Python 3.12 和锁定依赖
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{Path.home()}/.local/bin:" + os.environ["PATH"]
%cd /kaggle/working/MuVisual-Workflow
!uv python install 3.12
!uv sync --locked --python 3.12

In [ ]:
# 从 Kaggle Add-ons > Secrets 读取 HF_TOKEN
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
assert token and token.startswith("hf_"), "Kaggle Secret HF_TOKEN 不存在或格式错误"

os.environ["HF_TOKEN"] = token
os.environ["HF_HOME"] = str(CACHE_DIR / "huggingface")
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "huggingface" / "hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
os.environ["AUDIO_SEPARATOR_MODEL_DIR"] = str(CACHE_DIR / "separator")

print("HF_TOKEN 和模型缓存已配置")

In [ ]:
# 验证 Hugging Face 权限和 CUDA
%cd /kaggle/working/MuVisual-Workflow
!uv run python -c "from huggingface_hub import HfApi; api=HfApi(); print('HF user:', api.whoami()['name']); api.auth_check('MuScriptor/muscriptor-medium'); print('MuScriptor access: OK')"
!uv run python -c "import torch; print('Torch:', torch.__version__); print('CUDA wheel:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"

In [ ]:
# 递归批量处理 /kaggle/input 下的所有支持音频
%cd /kaggle/working/MuVisual-Workflow
!uv run muvisual --input "{INPUT_DIR}" --output "{OUTPUT_DIR}" --device cuda

In [ ]:
# 打包结果；也可通过 Save Version 将 /kaggle/working 作为 Notebook Output 保存
import shutil

archive = shutil.make_archive("/kaggle/working/muvisual-output", "zip", OUTPUT_DIR)
print(f"结果目录：{OUTPUT_DIR}")
print(f"下载文件：{archive}")